### **Modelos color percepción**

In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import os
import time
from ipywidgets import interact, FloatSlider, Dropdown

# Etapa 1: Cargar imagen
def cargar_imagen(ruta_imagen):
    """Carga una imagen y la convierte a RGB."""
    img = cv2.imread(ruta_imagen)
    if img is None:
        raise FileNotFoundError(f"No se pudo cargar la imagen en {ruta_imagen}")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img_rgb

# Etapa 2: Conversiones de espacio de color
def rgb_a_hsv(img_rgb):
    """Convierte una imagen RGB a HSV."""
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

def rgb_a_lab(img_rgb):
    """Convierte una imagen RGB a CIE Lab."""
    return cv2.cvtColor(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2Lab) / 255.0

# Etapa 3: Simulaciones de alteraciones visuales
def rgb_a_lms(img_rgb):
    """Convierte de RGB a LMS manualmente usando una matriz estándar."""
    matriz_rgb_a_lms = np.array([
        [0.4002, 0.7076, -0.0808],
        [-0.2263, 1.1653, 0.0457],
        [0.0, 0.0, 0.9182]
    ])
    img = img_rgb / 255.0
    alto, ancho, _ = img.shape
    img_flat = img.reshape(-1, 3)
    lms = np.dot(img_flat, matriz_rgb_a_lms.T)
    return lms.reshape(alto, ancho, 3)

def lms_a_rgb(lms):
    """Convierte de LMS a RGB manualmente usando la matriz inversa."""
    matriz_lms_a_rgb = np.array([
        [1.8601, -1.1295, 0.2199],
        [0.3612, 0.6388, -0.0000],
        [0.0000, 0.0000, 1.0891]
    ])
    alto, ancho, _ = lms.shape
    lms_flat = lms.reshape(-1, 3)
    rgb = np.dot(lms_flat, matriz_lms_a_rgb.T)
    rgb = np.clip(rgb, 0, 1) * 255
    return rgb.reshape(alto, ancho, 3).astype(np.uint8)

def simular_protanopia(img_rgb):
    """Simula protanopía (daltonismo) usando el espacio LMS."""
    lms = rgb_a_lms(img_rgb)
    matriz_protanopia = np.array([
        [0, 2.02344, -2.52581],
        [0, 1, 0],
        [0, 0, 1]
    ])
    alto, ancho, _ = lms.shape
    lms_flat = lms.reshape(-1, 3)
    lms_protanopia = np.dot(lms_flat, matriz_protanopia.T)
    lms_protanopia = lms_protanopia.reshape(alto, ancho, 3)
    return lms_a_rgb(lms_protanopia)

def simular_deuteranopia(img_rgb):
    """Simula deuteranopía (daltonismo) usando el espacio LMS."""
    lms = rgb_a_lms(img_rgb)
    matriz_deuteranopia = np.array([
        [1, 0, 0],
        [0.494207, 0, 1.24827],
        [0, 0, 1]
    ])
    alto, ancho, _ = lms.shape
    lms_flat = lms.reshape(-1, 3)
    lms_deuteranopia = np.dot(lms_flat, matriz_deuteranopia.T)
    lms_deuteranopia = lms_deuteranopia.reshape(alto, ancho, 3)
    return lms_a_rgb(lms_deuteranopia)

def simular_baja_luz(img_rgb, brillo=0.5):
    """Simula condiciones de baja luz reduciendo el brillo."""
    return np.clip(img_rgb * brillo, 0, 255).astype(np.uint8)

# Etapa 4: Transformaciones personalizadas
def aplicar_filtro_calido(img_rgb, calidez=0.2):
    """Aplica un filtro cálido ajustando los canales RGB."""
    img = img_rgb.copy()
    img[:, :, 0] = np.clip(img[:, :, 0] * (1 + calidez), 0, 255)
    img[:, :, 2] = np.clip(img[:, :, 2] * (1 - calidez * 0.5), 0, 255)
    return img

def aplicar_monocromo(img_rgb):
    """Convierte la imagen a monocromo (escala de grises)."""
    gris = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    return cv2.cvtColor(gris, cv2.COLOR_GRAY2RGB)

# Etapa 5: Visualización
def visualizar_canales(img, titulo, nombres_canales, ruta_salida):
    """Visualiza los canales individuales de un espacio de color."""
    fig, ejes = plt.subplots(1, len(nombres_canales), figsize=(15, 5))
    for i, eje in enumerate(ejes):
        eje.imshow(img[:, :, i], cmap='gray')
        eje.set_title(f'{titulo} - {nombres_canales[i]}')
        eje.axis('off')
    plt.tight_layout()
    fig.savefig(ruta_salida)
    plt.close(fig)

# Etapa 6: Generar GIFs
def generar_gif(imagenes, titulos, ruta_salida):
    """Genera un GIF animado a partir de imágenes y títulos."""
    with imageio.get_writer(ruta_salida, mode='I', duration=5.0, loop=0) as writer:
        for img, titulo in zip(imagenes, titulos):
            fig, eje = plt.subplots(figsize=(8, 8))
            if len(img.shape) == 2:
                eje.imshow(img, cmap='gray')
            else:
                eje.imshow(img)
            eje.set_title(titulo)
            eje.axis('off')
            fig.canvas.draw()
            buffer = fig.canvas.buffer_rgba()
            imagen = np.frombuffer(buffer, dtype=np.uint8).reshape(fig.canvas.get_width_height()[::-1] + (4,))
            imagen_rgb = imagen[..., :3]
            writer.append_data(imagen_rgb)
            plt.close(fig)
            time.sleep(0.5)  # Retardo adicional para asegurar procesamiento

# Etapa 7: Interactividad con sliders
def aplicar_transformacion(modo, brillo, calidez):
    """Aplica transformaciones dinámicas con controles interactivos."""
    img = img_rgb.copy()
    if modo == 'HSV':
        img = rgb_a_hsv(img)
        img = img[:, :, 0]  # Mostrar canal Matiz
        titulo = 'HSV - Matiz'
    elif modo == 'CIE Lab':
        img = rgb_a_lab(img)
        img = img[:, :, 0]  # Mostrar canal L
        titulo = 'CIE Lab - L'
    elif modo == 'Protanopía':
        img = simular_protanopia(img)
        titulo = 'Protanopía'
    elif modo == 'Deuteranopía':
        img = simular_deuteranopia(img)
        titulo = 'Deuteranopía'
    elif modo == 'Baja Luz':
        img = simular_baja_luz(img, brillo)
        titulo = f'Baja Luz (Brillo: {brillo:.1f})'
    elif modo == 'Filtro Cálido':
        img = aplicar_filtro_calido(img, calidez)
        titulo = f'Filtro Cálido (Calidez: {calidez:.2f})'
    elif modo == 'Monocromo':
        img = aplicar_monocromo(img)
        titulo = 'Monocromo'
    else:
        titulo = 'RGB Original'
    plt.figure(figsize=(8, 8))
    plt.imshow(img, cmap='gray' if modo in ['HSV', 'CIE Lab'] else None)
    plt.title(titulo)
    plt.axis('off')
    plt.show()

# Ejecución principal
if __name__ == '__main__':
    # Configuración
    ruta_imagen = 'python/imagen_ejemplo.jpg'
    directorio_salida = 'python/resultados'
    os.makedirs(directorio_salida, exist_ok=True)

    # Etapa 1: Cargar imagen
    img_rgb = cargar_imagen(ruta_imagen)

    # Etapa 2: Conversiones de espacio de color
    img_hsv = rgb_a_hsv(img_rgb)
    img_lab = rgb_a_lab(img_rgb) * 255.0

    # Etapa 3: Simulaciones de alteraciones visuales
    img_protanopia = simular_protanopia(img_rgb)
    img_deuteranopia = simular_deuteranopia(img_rgb)
    img_baja_luz = simular_baja_luz(img_rgb)

    # Etapa 4: Transformaciones personalizadas
    img_calido = aplicar_filtro_calido(img_rgb)
    img_monocromo = aplicar_monocromo(img_rgb)

    # Etapa 5: Visualización de canales
    visualizar_canales(img_hsv, 'HSV', ['Matiz', 'Saturación', 'Valor'], f'{directorio_salida}/canales_hsv.png')
    visualizar_canales(img_lab, 'CIE Lab', ['L', 'a', 'b'], f'{directorio_salida}/canales_lab.png')

    # Etapa 6: Generar GIFs
    generar_gif(
        [img_rgb, img_hsv[:, :, 0], img_hsv[:, :, 1], img_hsv[:, :, 2], img_lab[:, :, 0], img_lab[:, :, 1], img_lab[:, :, 2]],
        ['RGB Original', 'HSV - Matiz', 'HSV - Saturación', 'HSV - Valor', 'CIE Lab - L', 'CIE Lab - a', 'CIE Lab - b'],
        f'{directorio_salida}/canales_espacio_color.gif'
    )
    generar_gif(
        [img_rgb, img_protanopia, img_deuteranopia, img_baja_luz, img_calido, img_monocromo],
        ['RGB Original', 'Protanopía', 'Deuteranopía', 'Baja Luz', 'Filtro Cálido', 'Monocromo'],
        f'{directorio_salida}/alteraciones_visuales.gif'
    )

    # Etapa 7: Interactividad
    interact(aplicar_transformacion,
             modo=Dropdown(options=['RGB', 'HSV', 'CIE Lab', 'Protanopía', 'Deuteranopía', 'Baja Luz', 'Filtro Cálido', 'Monocromo'], value='RGB', description='Modo'),
             brillo=FloatSlider(min=0.1, max=2.0, step=0.1, value=0.5, description='Brillo'),
             calidez=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.2, description='Calidez'))

interactive(children=(Dropdown(description='Modo', options=('RGB', 'HSV', 'CIE Lab', 'Protanopía', 'Deuteranop…